# 03 — PEFT with LoRA: learn the private routing taxonomy

This is the practical single-GPU training path. It converts the pinned BF16 checkpoint once, trains a small LoRA adapter with NVIDIA's Nemotron 3.5 Lightning Megatron-Bridge recipe, merges the adapter for ordinary Hugging Face inference, and measures the exact accuracy delta against Notebook 02's local BF16 baseline. It then asks the practical deployment question: did specialized Lightning reach or beat the hosted Nemotron Ultra reference on the same private-routing examples?

**Target:** one H100 80 GB; 512-token packed sequences; at most 40 optimizer steps. NVIDIA's official single-H100 cookbook reports about 79 GB peak at 2K sequence length, so this shorter workshop profile leaves more activation headroom.

**Required runtime:** run this notebook from the NeMo container Jupyter started by `launchable/setup.sh`. On Brev, open the custom Secure Link on host port **8889**, not host-managed Jupyter on 8888. A repository `.venv` kernel is API-only and does not contain the CUDA training stack.

In [ ]:
from pathlib import Path
import gc, json, os, subprocess, sys, time

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'artifacts/data/banking77'
BASELINE_PATH = ROOT / 'artifacts/evaluation/baseline_local_bf16.json'
CLOUD_BASELINE_GLOB = 'baseline_api_*_nvfp4_*_per_label.json'
MEGATRON_BASE = Path('/workspace/storage/checkpoints/lightning35-megatron')
LORA_ROOT = Path('/workspace/storage/checkpoints/banking77-lora')
MERGED_MODEL = Path('/workspace/storage/checkpoints/banking77-lora-hf')
RUN_TRAINING = True
RUN_MERGE = True
RUN_EVALUATION = True
print('Repository:', ROOT)
print('Python:', sys.executable)

## 1. Guardrails and reproducibility

The cell fails early when the GPU allocation is unsuitable or Notebook 02 has not frozen the exact BF16 baseline. Data preparation is idempotent; rerunning it with the same manifest does not silently resample.

In [ ]:
subprocess.run([sys.executable, 'scripts/preflight.py', '--profile', 'peft'], check=True)
import torch
free_gib, total_gib = (value / 1024**3 for value in torch.cuda.mem_get_info())
if free_gib < 70:
    raise RuntimeError(f'Only {free_gib:.1f} GiB is free. Release or shut down Notebook 02 before PEFT.')
print(f'GPU memory available: {free_gib:.1f}/{total_gib:.1f} GiB')
assert BASELINE_PATH.exists(), 'Run Notebook 02 through the local BF16 baseline first.'
assert (DATA_DIR / 'manifest.json').exists(), 'Run Notebook 01 or 02 data preparation first.'
manifest = json.loads((DATA_DIR / 'manifest.json').read_text())
baseline = json.loads(BASELINE_PATH.read_text())
print('Frozen baseline:', {k: baseline[k] for k in ('n', 'accuracy', 'macro_accuracy', 'valid_code_rate')})
print('Training rows:', manifest['counts']['training'])
cloud_baselines = {}
for cloud_path in sorted((ROOT / 'artifacts/evaluation').glob(CLOUD_BASELINE_GLOB)):
    cloud = json.loads(cloud_path.read_text())
    model_id = cloud['model']
    api_profile = cloud.get('api_profile', 'public')
    prompt_mode = cloud.get('prompt_mode', 'unspecified')
    prompt_version = cloud.get('prompt_protocol_version', 'unspecified')
    demonstration_count = cloud.get('demonstrations_per_request', 0)
    condition = (
        f'{api_profile}::{model_id}::{prompt_mode}::v{prompt_version}::'
        f'{demonstration_count}d'
    )
    if condition not in cloud_baselines or cloud['n'] > cloud_baselines[condition]['n']:
        cloud_baselines[condition] = cloud
for condition, cloud in cloud_baselines.items():
    print('Context-only hosted NVFP4 target:', condition, {
        key: cloud[key] for key in ('n', 'accuracy', 'macro_accuracy', 'valid_code_rate')
    })
if not cloud_baselines:
    print('Notebook 01 API reports not found; PEFT can still use the required local BF16 baseline.')

## 2. One-time checkpoint conversion

Megatron-Bridge trains from its distributed checkpoint format. Conversion is CPU-initialized and idempotent; the result can be reused by both PEFT and full SFT. Expect roughly another model-sized checkpoint on disk.

In [ ]:
from huggingface_hub import snapshot_download
from nemotron_ft_lab.constants import MODEL_ID, MODEL_REVISION

PINNED_HF_MODEL = Path(snapshot_download(
    repo_id=MODEL_ID, revision=MODEL_REVISION, local_files_only=True,
))
print('Pinned local snapshot:', PINNED_HF_MODEL)

convert_cmd = [
    sys.executable, 'scripts/convert_checkpoint.py',
    '--hf-model', MODEL_ID, '--revision', MODEL_REVISION,
    '--output', str(MEGATRON_BASE),
]
if RUN_TRAINING:
    started = time.perf_counter()
    subprocess.run(convert_cmd, check=True)
    print(f'Conversion stage: {(time.perf_counter() - started) / 60:.1f} min')
else:
    print('Would run:', ' '.join(convert_cmd))

## 3. Construct and run the official LoRA recipe

The recipe owns model-specific target modules, including compatible Mamba, attention, routed-expert, and shared-expert projections. The lab changes only paths, topology, sequence length, schedule, and LoRA capacity. On one GPU it uses the checkpoint's single physical MTP head, matching NVIDIA's single-H100 cookbook constraint.

In [ ]:
N_GPUS = torch.cuda.device_count()
train_cmd = [
    'torchrun', f'--nproc-per-node={N_GPUS}', 'scripts/train_peft.py',
    '--megatron-checkpoint', str(MEGATRON_BASE),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(LORA_ROOT),
    '--sequence-length', '512',
    '--global-batch-size', '16',
    '--max-steps', '40',
    '--learning-rate', '1e-4',
    '--lora-rank', '16',
]
print('Launch:', ' '.join(train_cmd))
if RUN_TRAINING:
    started = time.perf_counter()
    subprocess.run(train_cmd, check=True)
    print(f'PEFT stage: {(time.perf_counter() - started) / 60:.1f} min')

In [ ]:
iteration_marker = LORA_ROOT / 'latest_checkpointed_iteration.txt'
if RUN_TRAINING:
    assert iteration_marker.exists(), f'Missing adapter marker: {iteration_marker}'
    latest_step = int(iteration_marker.read_text().strip())
    adapter_checkpoint = LORA_ROOT / f'iter_{latest_step:07d}'
    print('Adapter checkpoint:', adapter_checkpoint)
else:
    adapter_checkpoint = Path('/path/to/adapter')

## 4. Merge to a standard Hugging Face checkpoint

Megatron's adapter checkpoint is compact, but the comparison uses a merged model so the same Hugging Face generation code evaluates baseline and tuned weights. Keep at least 300 GB disk for the HF cache, converted base, adapter, and merged export.

In [ ]:
BRIDGE_DIR = Path(os.environ.get('MEGATRON_BRIDGE_DIR', '/workspace/storage/Megatron-Bridge'))
merge_script = BRIDGE_DIR / 'examples/peft/merge_lora.py'
merge_cmd = [
    'torchrun', '--nproc-per-node=1', str(merge_script),
    '--lora-checkpoint', str(adapter_checkpoint),
    '--hf-model-path', str(PINNED_HF_MODEL),
    '--output', str(MERGED_MODEL), '--cpu',
]
print('Merge:', ' '.join(merge_cmd))
if RUN_MERGE:
    started = time.perf_counter()
    subprocess.run(merge_cmd, check=True)
    assert (MERGED_MODEL / 'config.json').exists()
    print(f'Merge stage: {(time.perf_counter() - started) / 60:.1f} min')

## 5. Evaluate the unchanged holdout

A falling training loss is a diagnostic, not the outcome. Success requires a positive exact-accuracy delta on the same official-test IDs used for the baseline. Because the route codes are class-balanced in this bundle, macro and micro accuracy should be similar.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from nemotron_ft_lab.data import read_jsonl
from nemotron_ft_lab.evaluation import (
    generate_predictions, paired_accuracy_comparison, save_report, score_predictions,
)

all_eval_rows = read_jsonl(DATA_DIR / 'evaluation.jsonl')
eval_by_id = {row['example_id']: row for row in all_eval_rows}
baseline_ids = [row['example_id'] for row in baseline['rows']]
missing_baseline_ids = set(baseline_ids).difference(eval_by_id)
assert not missing_baseline_ids, f'Baseline IDs missing from frozen data: {sorted(missing_baseline_ids)[:3]}'
eval_rows = [eval_by_id[item] for item in baseline_ids]
if RUN_EVALUATION:
    tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL, trust_remote_code=True)
    tuned_model = AutoModelForCausalLM.from_pretrained(
        MERGED_MODEL, trust_remote_code=True, dtype=torch.bfloat16,
        device_map='auto', low_cpu_mem_usage=True,
    )
    started = time.perf_counter()
    tuned_rows = generate_predictions(tuned_model, tokenizer, eval_rows, batch_size=8)
    tuned = score_predictions(tuned_rows)
    tuned['wall_time_seconds'] = time.perf_counter() - started
    save_report(
        ROOT / 'artifacts/evaluation/peft.json', tuned,
        model=str(MERGED_MODEL), run_type='lora-peft'
    )
    del tuned_model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
else:
    tuned = json.loads((ROOT / 'artifacts/evaluation/peft.json').read_text())

In [ ]:
comparison = paired_accuracy_comparison(baseline, tuned)
comparison.update({
    'baseline_accuracy': baseline['accuracy'],
    'peft_accuracy': tuned['accuracy'],
    'baseline_valid_code_rate': baseline['valid_code_rate'],
    'peft_valid_code_rate': tuned['valid_code_rate'],
})
print(json.dumps(comparison, indent=2))
if comparison['absolute_accuracy_gain'] <= 0:
    print('No held-out gain was demonstrated. Do not claim success; inspect loss, formatting, and confused labels.')

## 6. Did specialized Lightning reach or beat hosted Ultra?

This is a task-specific deployment comparison, not an overall model ranking. Each cloud report may use the 77-row smoke profile or the 231-row extended profile. The cell subsets tuned Lightning to those exact IDs, reports tuned-minus-cloud accuracy with a paired 95% bootstrap interval, and saves the comparison. A confidence interval crossing zero is **inconclusive**, not proof that the models are equivalent.

In [ ]:
def target_conclusion(result, practical_margin=0.02):
    low, high = result['paired_bootstrap_95ci']
    gain = result['absolute_accuracy_gain']
    if low > 0:
        return 'tuned Lightning is better on this task at the paired 95% level'
    if high < 0:
        return 'tuned Lightning is worse on this task at the paired 95% level'
    if abs(gain) <= practical_margin:
        return 'point estimates are within 2 points; the paired result is inconclusive'
    return 'the observed gap is inconclusive at this evaluation size'

tuned_by_id = {row['example_id']: row for row in tuned['rows']}
cloud_target_results = {}
for condition, cloud in cloud_baselines.items():
    model_id = cloud['model']
    prompt_mode = cloud.get('prompt_mode', 'unspecified')
    cloud_ids = [row['example_id'] for row in cloud['rows']]
    missing = set(cloud_ids).difference(tuned_by_id)
    if missing:
        print(f'{condition}: IDs differ from this frozen bundle; skipped.')
        continue
    tuned_matched = score_predictions([tuned_by_id[item] for item in cloud_ids])
    result = paired_accuracy_comparison(cloud, tuned_matched)
    result.update({
        'cloud_accuracy': cloud['accuracy'],
        'api_profile': cloud.get('api_profile', 'public'),
        'endpoint': cloud.get('endpoint', 'unspecified'),
        'cloud_model': model_id,
        'prompt_mode': prompt_mode,
        'prompt_protocol_version': cloud.get('prompt_protocol_version', 'unspecified'),
        'demonstrations_per_request': cloud.get('demonstrations_per_request', 0),
        'tuned_lightning_accuracy_on_shared_ids': tuned_matched['accuracy'],
        'conclusion': target_conclusion(result),
        'interpretation': 'tuned Lightning BF16-derived minus hosted cloud NVFP4',
    })
    cloud_target_results[condition] = result
    print(f'\n{condition}')
    print(json.dumps(result, indent=2))

target_path = ROOT / 'artifacts/evaluation/peft_vs_cloud_targets.json'
target_path.write_text(json.dumps(cloud_target_results, indent=2) + '\n')
print('Saved:', target_path)

## What PEFT changed—and what it did not prove

A positive paired local-before/local-after delta shows that LoRA improved the pinned BF16 checkpoint on the private taxonomy while leaving its base weights frozen. Beating hosted Ultra would additionally show that a small specialized model can outperform a much larger general model on this narrow private task; it would not imply that tuned Lightning is generally stronger. Notebook 04 explains why updating every parameter is outside the ordinary one-GPU Brev exercise.